# Notebook 3 : SQL

In [ ]:
# Décommenter la ligne suivante pour installer ibis
#%pip install 'ibis-framework[sqlite]'

In [5]:
import sqlite3

import pandas as pd
import ibis

from ibis import _

ibis.options.interactive = True

query_tables = "SELECT name FROM sqlite_master WHERE type='table'"

## STAR

Nous considérons les données des stations de vélos en libre service [STAR](https://www.star.fr/) de Rennes Métropole. Une copie de la base SQLite est disponible dans le fichier `star.db`. Nous utilisons d'abord Pandas pour répondre aux questions, puis Ibis.

1. Se connecter à la base de données et afficher la liste des tables à l'aide de la fonction `read_sql` de Pandas et de la requête `query_tables`.

In [7]:
con = sqlite3.connect("data/star.db")

print(
    pd.read_sql(query_tables, con)
    )


        name
0  Topologie
1       Etat


2. Récupérer le contenu de la table `Etat` dans un dataframe et afficher la liste des variables disponibles. Même question pour la table `Topologie`.

In [12]:
df_etat = pd.read_sql("SELECT * FROM Etat", con)
df_etat.dtypes
df_etat.head(10)

,id,nom,latitude,longitude,etat,nb_emplacements,emplacements_disponibles,velos_disponibles,date,data
0,1,République,48.110026,-1.678037,En fonctionnement,30,25,5,1.524646e+09,2018-04-25 08:47:04
1,2,Mairie,48.111624,-1.678757,En fonctionnement,24,6,18,1.524646e+09,2018-04-25 08:47:04
2,3,Champ Jacquet,48.112764,-1.680062,En fonctionnement,24,8,16,1.524646e+09,2018-04-25 08:47:04
3,10,Musée Beaux-Arts,48.109601,-1.674080,En fonctionnement,16,4,12,1.524646e+09,2018-04-25 08:47:04
4,12,TNB,48.107748,-1.673711,En fonctionnement,28,16,12,1.524646e+09,2018-04-25 08:47:04
5,14,Laënnec,48.106847,-1.665817,En fonctionnement,16,3,13,1.524646e+09,2018-04-25 08:47:04
6,17,Charles de Gaulle,48.105111,-1.677119,En fonctionnement,24,17,7,1.524646e+09,2018-04-25 08:47:04
7,20,Pont de Nantes,48.102015,-1.684015,En fonctionnement,20,9,11,1.524646e+09,2018-04-25 08:47:04
8,22,Oberthur,48.113550,-1.661858,En fonctionnement,20,13,7,1.524646e+09,2018-04-25 08:47:04
9,25,Office de Tourisme,48.110294,-1.683106,En fonctionnement,10,6,4,1.524646e+09,2018-04-25 08:47:04


3. Sélectionner l'identifiant `id`, le nom `nom` et l'identifiant de la station la plus proche `id_proche_1` depuis la table `Topologie`.

In [13]:
df_topo = pd.read_sql("SELECT id,nom,id_proche_1 FROM Topologie", con)
df_topo.dtypes
df_topo.head(10)

,id,nom,id_proche_1
0,1,République,2
1,2,Mairie,1
2,3,Champ Jacquet,2
3,10,Musée Beaux-Arts,12
4,12,TNB,10
5,14,Laënnec,35
6,17,Charles de Gaulle,16
7,20,Pont de Nantes,43
8,22,Oberthur,44
9,25,Office de Tourisme,24


4. Faire une jointure sur la table précédente pour créer une table qui contient la liste des stations avec l'identifiant, le nom et le nom de la station la plus proche associée à l'identifiant `id_proche_1`. Les variables utilisées comme clés sont différents, penser à utiliser les arguments `left_on` et `right_on` de la méthode `merge`.

In [16]:
df_topo.merge(df_etat[["id", "nom",'etat']].rename(columns={"nom": "nom_plus_proche"}),left_on="id_proche_1", right_on='id',how='left')

,id_x,nom,id_proche_1,id_y,nom_plus_proche,etat
0,1,République,2,2.0,Mairie,En fonctionnement
1,2,Mairie,1,1.0,République,En fonctionnement
2,3,Champ Jacquet,2,2.0,Mairie,En fonctionnement
3,10,Musée Beaux-Arts,12,12.0,TNB,En fonctionnement
4,12,TNB,10,10.0,Musée Beaux-Arts,En fonctionnement
...,...,...,...,...,...,...
78,62,Clemenceau,63,63.0,Henri Fréville,En fonctionnement
79,66,Bréquigny Piscine,65,NaN,NaN,NaN
80,69,Champs Manceaux,66,66.0,Bréquigny Piscine,En fonctionnement
81,85,La Courrouze,20,20.0,Pont de Nantes,En fonctionnement


5. Ajouter à la table précédente la distance entre la station et la station la plus proche.

In [ ]:
df=df_topo.merge(df_etat[["id", "nom",'etat']].rename(columns={"nom": "nom_plus_proche"}),left_on="id_proche_1", right_on='id',how='left')


6. Créer une table avec le nom des trois stations les plus proches du point GPS *(48.1179151,-1.7028661)* classées par ordre de distance et le nombre de vélos disponibles dans ces stations.

7. Reprendre les questions précédentes en utilisant le module `ibis`. Pour les jointures, utiliser la méthode `left_join`.

In [17]:
#1
con = ibis.sqlite.connect()
con = ibis.sqlite.connect("data/star.db")
print(con.tables)

Tables
------
- Etat
- Topologie


In [19]:
#2
df_etat = con.table("Etat")
print(df_etat )


┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━
┃ id    ┃ nom                ┃ latitude  ┃ longitude ┃ etat              ┃ nb_emplacements ┃ emplacements_disponibl
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━
│ int64 │ string             │ float64   │ float64   │ string            │ int64           │ int64                 
├───────┼────────────────────┼───────────┼───────────┼───────────────────┼─────────────────┼───────────────────────
│     1 │ République         │ 48.110026 │ -1.678037 │ En fonctionnement │              30 │                       
│     2 │ Mairie             │ 48.111624 │ -1.678757 │ En fonctionnement │              24 │                       
│     3 │ Champ Jacquet      │ 48.112764 │ -1.680062 │ En fonctionnement │              24 │                       
│    10 │ Musée Beaux-Arts   │ 48.109601 │ -1.674080 │ En fonctionnement

In [ ]:
#3
print(con.table("Topologie").select('id','nom','id_proche_1'))

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ id    ┃ nom                ┃ id_proche_1 ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ int64 │ string             │ int64       │
├───────┼────────────────────┼─────────────┤
│     1 │ République         │           2 │
│     2 │ Mairie             │           1 │
│     3 │ Champ Jacquet      │           2 │
│    10 │ Musée Beaux-Arts   │          12 │
│    12 │ TNB                │          10 │
│    14 │ Laënnec            │          35 │
│    17 │ Charles de Gaulle  │          16 │
│    20 │ Pont de Nantes     │          43 │
│    22 │ Oberthur           │          44 │
│    25 │ Office de Tourisme │          24 │
│     … │ …                  │           … │
└───────┴────────────────────┴─────────────┘


In [34]:
#4
df_topo=con.table("Topologie")
df_etat=con.table("Etat")

df=(
    df_topo
    .left_join(df_etat.select("id", "nom",'etat','longitude','latitude').rename({"nom_plus_proche": "nom",'longitude_plus_proche':'longitude','latitude_plus_proche':'latitude'}), df_topo.id_proche_1 == df_etat.id) # Jointure
    )

print(df.select("id", "nom",'longitude','latitude','id_proche_1','etat',"nom_plus_proche",'longitude_plus_proche','latitude_plus_proche'))

┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━
┃ id    ┃ nom                ┃ longitude ┃ latitude  ┃ id_proche_1 ┃ etat              ┃ nom_plus_proche    ┃ longi
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━
│ int64 │ string             │ float64   │ float64   │ int64       │ string            │ string             │ float
├───────┼────────────────────┼───────────┼───────────┼─────────────┼───────────────────┼────────────────────┼──────
│     1 │ République         │ -1.678037 │ 48.110026 │           2 │ En fonctionnement │ Mairie             │      
│     2 │ Mairie             │ -1.678757 │ 48.111624 │           1 │ En fonctionnement │ République         │      
│     3 │ Champ Jacquet      │ -1.680062 │ 48.112764 │           2 │ En fonctionnement │ Mairie             │      
│    10 │ Musée Beaux-Arts   │ -1.674080 │ 48.109601 │          12 │ En 

In [ ]:
#5

df=df.mutate(distance = df.adresse_numero + " " + topologie.adresse_voie)

8. (*Bonus*) Écrire des requêtes SQL pour obtenir les résultats demandés dans les questions 3 à 6. La fonction `to_sql` pourra être utilisée pour de l'aide.

## Musique

Le dépôt GitHub [lerocha/chinook-database](https://github.com/lerocha/chinook-database) met à disposition des bases de données de bibliothèques musicales. Une copie de la base SQLite est disponible dans le fichier `chinook.db`.

1. Utiliser le module `ibis` pour vous connecter à la base de données et explorer les tables formant le jeu de données pour le découvrir. En particulier, remarquer comment les tables `Playlist`, `PlaylistTrack` et `Track` sont liées entre elles.

2. Quelles sont les playlists qui contiennent le plus de pistes ?

3. Construire une table contenant les informations suivantes sur la playlist `Classical` : le titre de chaque piste ainsi que le titre de l'album dont cette piste est tirée.

4. (*Bonus*) Écrire une requête SQL donnant le résultat de la question précédente. La fonction `to_sql` pourra être utilisée pour de l'aide.